# Power Analysis: Phase 1 vs Phase 2 Final (plus Phase-2 Reasoning Shift)

This notebook now does exactly this split:

1. **Phase 1 vs Phase 2-final (independent participants)**
- Top-choice power analysis
- Rank-based power analysis

2. **Phase 2 only (within-person)**
- Reasoning change via L1 distance from `initial_reasoning` to `final_reasoning`

Frequentist defaults:
- `alpha = 0.05`
- target power `= 0.80`


In [1]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm


In [2]:
# Config
PHASE1_EXTRACT_PATH = Path('../data/responses/phase1_hst_dbf516d28d66_extractions.json')
PHASE2_EXTRACT_PATH = Path('../data/responses/phase2_hst_f3f99c5cc524_extractions.json')

ALPHA = 0.05
TARGET_POWER = 0.80


In [3]:
def load_json_rows(path: Path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        if isinstance(payload.get('data'), list):
            return payload['data']
        if isinstance(payload.get('participants'), list):
            return payload['participants']
    raise ValueError(f'Unsupported JSON shape in {path}')

phase1_rows = load_json_rows(PHASE1_EXTRACT_PATH)
phase2_rows = load_json_rows(PHASE2_EXTRACT_PATH)

phase1_df = pd.DataFrame(phase1_rows)
phase2_df = pd.DataFrame(phase2_rows)

# For across-phase comparisons, use phase1 vote_ranking vs phase2 final_vote_ranking
phase1_cmp = phase1_df[['user_id', 'vote_ranking', 'reasoning']].copy()
phase2_cmp = phase2_df[['user_id', 'final_vote_ranking', 'final_reasoning']].copy()
phase2_cmp = phase2_cmp.rename(columns={
    'final_vote_ranking': 'vote_ranking',
    'final_reasoning': 'reasoning',
})

print('Phase 1 n =', len(phase1_cmp))
print('Phase 2 final n =', len(phase2_cmp))


Phase 1 n = 12
Phase 2 final n = 18


In [4]:
def get_options(*frames: pd.DataFrame) -> list[str]:
    opts = set()
    for frame in frames:
        for ranking in frame['vote_ranking']:
            if isinstance(ranking, list):
                opts.update(ranking)
    return sorted(opts)

OPTIONS = get_options(phase1_cmp, phase2_cmp)
OPTIONS


['animal_rescue', 'community_clinic', 'food_pantry', 'urban_tree']

## A) Phase 1 vs Phase 2-final: Top-Choice Power (Independent Groups)


In [5]:
def cohens_h(p1: float, p2: float) -> float:
    p1 = min(max(p1, 1e-9), 1 - 1e-9)
    p2 = min(max(p2, 1e-9), 1 - 1e-9)
    return 2 * math.asin(math.sqrt(p1)) - 2 * math.asin(math.sqrt(p2))


def required_n_equal_groups(effect_size_abs: float, alpha: float = 0.05, power: float = 0.80) -> float:
    if effect_size_abs < 1e-12:
        return np.inf
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    return 2 * (z_alpha + z_beta) ** 2 / (effect_size_abs ** 2)


def top_choice(series):
    return series.apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)

p1_top = top_choice(phase1_cmp['vote_ranking'])
p2_top = top_choice(phase2_cmp['vote_ranking'])

top_rows = []
for opt in OPTIONS:
    p1 = float((p1_top == opt).mean())
    p2 = float((p2_top == opt).mean())
    h = abs(cohens_h(p1, p2))
    n_req = required_n_equal_groups(h, alpha=ALPHA, power=TARGET_POWER)
    top_rows.append({
        'option': opt,
        'phase1_top_share': p1,
        'phase2_final_top_share': p2,
        'effect_size_h_abs': h,
        'required_n_per_group': math.ceil(n_req) if np.isfinite(n_req) else np.inf,
    })

top_power_df = pd.DataFrame(top_rows).sort_values('required_n_per_group')
top_power_df


,option,phase1_top_share,phase2_final_top_share,effect_size_h_abs,required_n_per_group
3,urban_tree,0.333333,0.166667,0.389891,104
0,animal_rescue,0.083333,0.166667,0.255383,241
1,community_clinic,0.333333,0.388889,0.115744,1172
2,food_pantry,0.250000,0.277778,0.063045,3950


## B) Phase 1 vs Phase 2-final: Rank-Based Power (Independent Groups)


In [6]:
def ranking_to_positions(ranking: list[str], options: list[str]) -> dict[str, int]:
    pos = {opt: np.nan for opt in options}
    if isinstance(ranking, list):
        for i, opt in enumerate(ranking, start=1):
            pos[opt] = i
    return pos

rank_rows = []
for _, row in phase1_cmp.iterrows():
    pos = ranking_to_positions(row['vote_ranking'], OPTIONS)
    for opt, rank in pos.items():
        rank_rows.append({'phase': 'phase1', 'option': opt, 'rank': rank})
for _, row in phase2_cmp.iterrows():
    pos = ranking_to_positions(row['vote_ranking'], OPTIONS)
    for opt, rank in pos.items():
        rank_rows.append({'phase': 'phase2_final', 'option': opt, 'rank': rank})

rank_df = pd.DataFrame(rank_rows)


def cohens_d_independent(x1: np.ndarray, x2: np.ndarray) -> float:
    x1 = np.asarray(x1, dtype=float)
    x2 = np.asarray(x2, dtype=float)
    n1, n2 = len(x1), len(x2)
    if n1 < 2 or n2 < 2:
        return np.nan
    v1 = np.var(x1, ddof=1)
    v2 = np.var(x2, ddof=1)
    sp = math.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
    if sp == 0:
        return 0.0
    return (np.mean(x1) - np.mean(x2)) / sp

rank_rows_out = []
for opt in OPTIONS:
    x1 = rank_df[(rank_df['phase'] == 'phase1') & (rank_df['option'] == opt)]['rank'].dropna().to_numpy()
    x2 = rank_df[(rank_df['phase'] == 'phase2_final') & (rank_df['option'] == opt)]['rank'].dropna().to_numpy()
    d = abs(cohens_d_independent(x1, x2))
    n_req = required_n_equal_groups(d, alpha=ALPHA, power=TARGET_POWER)
    rank_rows_out.append({
        'option': opt,
        'phase1_mean_rank': float(np.mean(x1)),
        'phase2_final_mean_rank': float(np.mean(x2)),
        'effect_size_d_abs': d,
        'required_n_per_group': math.ceil(n_req) if np.isfinite(n_req) else np.inf,
    })

rank_power_df = pd.DataFrame(rank_rows_out).sort_values('required_n_per_group')
rank_power_df


,option,phase1_mean_rank,phase2_final_mean_rank,effect_size_d_abs,required_n_per_group
0,animal_rescue,2.750000,3.055556,0.299240,176
2,food_pantry,2.083333,1.944444,0.164909,578
1,community_clinic,2.166667,2.055556,0.106267,1391
3,urban_tree,3.000000,2.944444,0.043850,8164


## C) Phase 2 Only: Individual Ranking Movement Metrics

These metrics quantify how much each participant changed from initial to final ranking:

- `top_choice_switched` (0/1)
- `top_choice_displacement` (how far their initial #1 moved in final ranking)
- `footrule_distance` (sum of absolute rank changes across all options)
- `kendall_distance` (pairwise inversion count)
- `avg_abs_rank_shift` (average absolute rank change per option)

In [ ]:
phase2_rankings = phase2_df[['user_id', 'initial_vote_ranking', 'final_vote_ranking']].copy()


def kendall_distance(rank_a: list[str], rank_b: list[str], options: list[str]) -> int:
    pa = ranking_to_positions(rank_a, options)
    pb = ranking_to_positions(rank_b, options)
    d = 0
    for i in range(len(options)):
        for j in range(i + 1, len(options)):
            oi = options[i]
            oj = options[j]
            if np.isnan(pa[oi]) or np.isnan(pa[oj]) or np.isnan(pb[oi]) or np.isnan(pb[oj]):
                continue
            if (pa[oi] < pa[oj]) != (pb[oi] < pb[oj]):
                d += 1
    return d


def footrule_distance(rank_a: list[str], rank_b: list[str], options: list[str]) -> int:
    pa = ranking_to_positions(rank_a, options)
    pb = ranking_to_positions(rank_b, options)
    total = 0
    for o in options:
        if np.isnan(pa[o]) or np.isnan(pb[o]):
            continue
        total += int(abs(pa[o] - pb[o]))
    return total


movement_rows = []
for _, r in phase2_rankings.iterrows():
    init_rank = r['initial_vote_ranking'] if isinstance(r['initial_vote_ranking'], list) else []
    final_rank = r['final_vote_ranking'] if isinstance(r['final_vote_ranking'], list) else []

    init_top = init_rank[0] if init_rank else None
    final_top = final_rank[0] if final_rank else None
    switched = int(init_top != final_top) if (init_top and final_top) else np.nan

    p_init = ranking_to_positions(init_rank, OPTIONS)
    p_final = ranking_to_positions(final_rank, OPTIONS)

    if init_top in p_final and not np.isnan(p_final.get(init_top, np.nan)):
        top_displacement = int(abs(1 - p_final[init_top]))
    else:
        top_displacement = np.nan

    abs_shifts = []
    for o in OPTIONS:
        if np.isnan(p_init[o]) or np.isnan(p_final[o]):
            continue
        abs_shifts.append(abs(p_final[o] - p_init[o]))

    movement_rows.append({
        'user_id': r['user_id'],
        'initial_top_choice': init_top,
        'final_top_choice': final_top,
        'top_choice_switched': switched,
        'top_choice_displacement': top_displacement,
        'footrule_distance': footrule_distance(init_rank, final_rank, OPTIONS),
        'kendall_distance': kendall_distance(init_rank, final_rank, OPTIONS),
        'avg_abs_rank_shift': float(np.mean(abs_shifts)) if abs_shifts else np.nan,
    })

phase2_individual_movement_df = pd.DataFrame(movement_rows)
phase2_individual_movement_df

In [ ]:
movement_summary = {
    'n_participants': int(len(phase2_individual_movement_df)),
    'share_top_choice_switched': float(phase2_individual_movement_df['top_choice_switched'].mean()),
    'mean_top_choice_displacement': float(phase2_individual_movement_df['top_choice_displacement'].mean()),
    'mean_footrule_distance': float(phase2_individual_movement_df['footrule_distance'].mean()),
    'mean_kendall_distance': float(phase2_individual_movement_df['kendall_distance'].mean()),
    'mean_avg_abs_rank_shift': float(phase2_individual_movement_df['avg_abs_rank_shift'].mean()),
}

movement_summary

## D) Phase 2 Only: Individual Reasoning Shift (Embedding Cosine Shift)

This section is intentionally Phase 2 only.
Each participant gets a reasoning-shift score from:

`1 - cosine_similarity(embedding(initial_reasoning), embedding(final_reasoning))`

- Higher values mean larger semantic shift.
- We use sentence-transformer embeddings if available.
- If unavailable, we fall back to TF-IDF vectors so the notebook still runs locally.

Then we estimate required `n` for a one-sample mean-shift test against 0.


In [7]:
phase2_within = phase2_df[['user_id', 'initial_reasoning', 'final_reasoning']].copy()
print('Phase 2 within-person n =', len(phase2_within))


Phase 2 within-person n = 18


In [8]:
def build_embeddings(texts: list[str]):
    # Return (matrix, embedding_backend_name)
    try:
        from sentence_transformers import SentenceTransformer
        model_name = 'sentence-transformers/all-MiniLM-L6-v2'
        model = SentenceTransformer(model_name)
        emb = model.encode(texts, convert_to_numpy=True, normalize_embeddings=False)
        return np.asarray(emb, dtype=float), model_name
    except Exception:
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2), min_df=1)
        emb = vec.fit_transform(texts).toarray()
        return np.asarray(emb, dtype=float), 'tfidf_unigram_bigram_fallback'


def rowwise_cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    # cosine(x, y) = <x,y> / (||x||*||y||); if a row is all-zero, similarity is set to 0
    dot = np.sum(a * b, axis=1)
    na = np.linalg.norm(a, axis=1)
    nb = np.linalg.norm(b, axis=1)
    denom = na * nb
    sim = np.zeros(len(dot), dtype=float)
    nz = denom > 0
    sim[nz] = dot[nz] / denom[nz]
    return sim


texts_initial = phase2_within['initial_reasoning'].fillna('').tolist()
texts_final = phase2_within['final_reasoning'].fillna('').tolist()
all_texts = texts_initial + texts_final

all_emb, embedding_backend = build_embeddings(all_texts)
n = len(phase2_within)
emb_initial = all_emb[:n]
emb_final = all_emb[n:]

cos_sim = rowwise_cosine_similarity(emb_initial, emb_final)
cos_shift = 1.0 - np.clip(cos_sim, -1.0, 1.0)

phase2_within['reasoning_cosine_shift_within_person'] = cos_shift

print('Embedding backend:', embedding_backend)
phase2_within[['reasoning_cosine_shift_within_person']].describe()


Embedding backend: sentence-transformers/all-MiniLM-L6-v2


,reasoning_cosine_shift_within_person
count,18.000000
mean,0.495011
std,0.170829
min,0.170216
25%,0.390471
50%,0.491139
75%,0.618466
max,0.841556


In [9]:
def one_sample_d(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return np.nan
    sd = np.std(x, ddof=1)
    if sd == 0:
        return np.nan
    return float(np.mean(x) / sd)


def required_n_one_sample(effect_size_abs: float, alpha: float = 0.05, power: float = 0.80) -> float:
    if effect_size_abs < 1e-12:
        return np.inf
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    return (z_alpha + z_beta) ** 2 / (effect_size_abs ** 2)

x = phase2_within['reasoning_cosine_shift_within_person'].dropna().to_numpy()
d = abs(one_sample_d(x))
n_req = required_n_one_sample(d, alpha=ALPHA, power=TARGET_POWER)

reasoning_power_df = pd.DataFrame([
    {
        'metric': 'phase2_reasoning_cosine_shift_within_person',
        'embedding_backend': embedding_backend,
        'mean_value': float(np.mean(x)),
        'effect_size_d_abs': d,
        'required_n': math.ceil(n_req) if np.isfinite(n_req) else np.inf,
    }
])
reasoning_power_df


,metric,embedding_backend,mean_value,effect_size_d_abs,required_n
0,phase2_reasoning_cosine_shift_within_person,sentence-transformers/all-MiniLM-L6-v2,0.495011,2.897698,1


## E) Between-Design Proxies (Phase 1 vs Phase 2-final)

These are **not individual change** metrics (participants differ across phases).
They are group-level proxies that quantify how far distributions moved between designs:

- Full ranking distribution shift (Total Variation distance)
- Option-wise rank distribution shift (Wasserstein-1)
- Participant distance-to-consensus shift (relative to Phase 1 consensus)

In [ ]:
from scipy.stats import wasserstein_distance


def canonical_ranking_str(ranking: list[str]) -> str:
    return ' > '.join(ranking) if isinstance(ranking, list) else 'MISSING'


def build_rank_position_frame(frame: pd.DataFrame, phase_label: str) -> pd.DataFrame:
    rows = []
    for _, r in frame.iterrows():
        pos = ranking_to_positions(r['vote_ranking'], OPTIONS)
        row = {'phase': phase_label, 'user_id': r['user_id']}
        for o in OPTIONS:
            row[f'rank__{o}'] = pos[o]
        row['ranking_str'] = canonical_ranking_str(r['vote_ranking'])
        rows.append(row)
    return pd.DataFrame(rows)


p1_pos = build_rank_position_frame(phase1_cmp, 'phase1')
p2_pos = build_rank_position_frame(phase2_cmp, 'phase2_final')

# 1) Full ranking distribution shift via Total Variation distance
p1_counts = p1_pos['ranking_str'].value_counts(normalize=True)
p2_counts = p2_pos['ranking_str'].value_counts(normalize=True)
all_rankings = sorted(set(p1_counts.index) | set(p2_counts.index))

tv = 0.5 * sum(abs(float(p1_counts.get(r, 0.0)) - float(p2_counts.get(r, 0.0))) for r in all_rankings)

# 2) Option-wise Wasserstein distance of rank distributions
option_proxy_rows = []
for o in OPTIONS:
    x1 = p1_pos[f'rank__{o}'].dropna().to_numpy(dtype=float)
    x2 = p2_pos[f'rank__{o}'].dropna().to_numpy(dtype=float)
    w1 = float(wasserstein_distance(x1, x2))
    option_proxy_rows.append({
        'option': o,
        'phase1_mean_rank': float(np.mean(x1)),
        'phase2_final_mean_rank': float(np.mean(x2)),
        'mean_rank_delta': float(np.mean(x2) - np.mean(x1)),
        'phase1_sd_rank': float(np.std(x1, ddof=1)),
        'phase2_final_sd_rank': float(np.std(x2, ddof=1)),
        'wasserstein_rank_distance': w1,
    })

between_option_proxy_df = pd.DataFrame(option_proxy_rows).sort_values('wasserstein_rank_distance', ascending=False)

# 3) Participant distance to Phase-1 consensus ranking
phase1_consensus_mean_rank = {o: float(p1_pos[f'rank__{o}'].mean()) for o in OPTIONS}
consensus_order = sorted(OPTIONS, key=lambda o: (phase1_consensus_mean_rank[o], o))
consensus_pos = {o: i+1 for i, o in enumerate(consensus_order)}

def footrule_to_consensus(rank_row: pd.Series) -> float:
    s = 0.0
    for o in OPTIONS:
        r = rank_row[f'rank__{o}']
        if np.isnan(r):
            continue
        s += abs(float(r) - float(consensus_pos[o]))
    return s

p1_pos['dist_to_phase1_consensus'] = p1_pos.apply(footrule_to_consensus, axis=1)
p2_pos['dist_to_phase1_consensus'] = p2_pos.apply(footrule_to_consensus, axis=1)

between_proxy_summary = pd.DataFrame([
    {
        'metric': 'tv_distance_over_full_ranking_distribution',
        'value': float(tv),
    },
    {
        'metric': 'mean_dist_to_phase1_consensus__phase1',
        'value': float(p1_pos['dist_to_phase1_consensus'].mean()),
    },
    {
        'metric': 'mean_dist_to_phase1_consensus__phase2_final',
        'value': float(p2_pos['dist_to_phase1_consensus'].mean()),
    },
    {
        'metric': 'delta_dist_to_phase1_consensus__phase2_minus_phase1',
        'value': float(p2_pos['dist_to_phase1_consensus'].mean() - p1_pos['dist_to_phase1_consensus'].mean()),
    },
])

between_proxy_summary

In [ ]:
between_option_proxy_df

## Combined Summary


In [10]:
summary_rows = []

for _, r in top_power_df.iterrows():
    summary_rows.append({
        'analysis_block': 'phase1_vs_phase2_top_choice',
        'metric': r['option'],
        'effect_size': r['effect_size_h_abs'],
        'required_n_per_group_or_total': r['required_n_per_group'],
    })

for _, r in rank_power_df.iterrows():
    summary_rows.append({
        'analysis_block': 'phase1_vs_phase2_rank',
        'metric': r['option'],
        'effect_size': r['effect_size_d_abs'],
        'required_n_per_group_or_total': r['required_n_per_group'],
    })

for metric_name in ['top_choice_switched','top_choice_displacement','footrule_distance','kendall_distance','avg_abs_rank_shift']:
    summary_rows.append({
        'analysis_block': 'phase2_within_ranking_movement',
        'metric': metric_name,
        'effect_size': float(phase2_individual_movement_df[metric_name].mean()),
        'required_n_per_group_or_total': np.nan,
    })

for _, r in between_proxy_summary.iterrows():
    summary_rows.append({
        'analysis_block': 'between_design_proxy',
        'metric': r['metric'],
        'effect_size': float(r['value']),
        'required_n_per_group_or_total': np.nan,
    })

for _, r in between_option_proxy_df.iterrows():
    summary_rows.append({
        'analysis_block': 'between_design_proxy_option',
        'metric': r['option'],
        'effect_size': float(r['wasserstein_rank_distance']),
        'required_n_per_group_or_total': np.nan,
    })

for _, r in reasoning_power_df.iterrows():
    summary_rows.append({
        'analysis_block': 'phase2_within_reasoning_l1',
        'metric': r['metric'],
        'effect_size': r['effect_size_d_abs'],
        'required_n_per_group_or_total': r['required_n'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.sort_values(['analysis_block', 'required_n_per_group_or_total'])


,analysis_block,metric,effect_size,required_n_per_group_or_total
4,phase1_vs_phase2_rank,animal_rescue,0.299240,176
5,phase1_vs_phase2_rank,food_pantry,0.164909,578
6,phase1_vs_phase2_rank,community_clinic,0.106267,1391
7,phase1_vs_phase2_rank,urban_tree,0.043850,8164
0,phase1_vs_phase2_top_choice,urban_tree,0.389891,104
1,phase1_vs_phase2_top_choice,animal_rescue,0.255383,241
2,phase1_vs_phase2_top_choice,community_clinic,0.115744,1172
3,phase1_vs_phase2_top_choice,food_pantry,0.063045,3950
8,phase2_within_reasoning_l1,phase2_reasoning_cosine_shift_within_person,2.897698,1


## Notes

- Top-choice and rank sections use Phase 1 vs Phase 2-final independent-group analysis.
- Individual ranking movement section is Phase 2-only within-person (switches, displacement, footrule, Kendall).
- Between-design proxy section compares Phase 1 vs Phase 2-final distributions (not person-level change).
- Reasoning section is Phase 2-only within-person, using embedding cosine shift.
- If `sentence-transformers` is available, it uses `all-MiniLM-L6-v2`; otherwise it falls back to TF-IDF.
- If you want a more conservative planning endpoint, define a thresholded binary shift metric (e.g., shift > 0.15) and power that proportion.
